# 12 · From notebook to reusable classifier

**Level:** advanced workflow · **Prerequisites:** training/evaluation, lesson 11's confidence caveats.
**Teaching time:** 60–90 minutes; training/download time depends on hardware.

Train a compact Fashion-MNIST classifier, package its weights **and preprocessing**, reload it in a
fresh object, predict a saved image using a standalone script, export a fixed-shape program, and
measure latency. Finish with a capstone plan for your own dataset.

This is a **local deployment demonstration**, not an internet service. It teaches artifact contracts,
prediction correctness, and measurement; a production service also needs monitoring and operational testing.
No previous notebook's checkpoint is required.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


## Step 1 — Define the task and input contract

This model recognizes the ten Fashion-MNIST clothing categories in grayscale 28×28 images.
It does not recognize arbitrary objects in full-size photos. Store the class order, resize operation,
color conversion, and normalization with the weights: a correct network with incorrect preprocessing
can still produce incorrect predictions.


In [ ]:
import json
import copy
import time
from PIL import Image
from torchvision import transforms

TRAIN_LIMIT = 128 if SMOKE else (4000 if QUICK else None)
VAL_LIMIT = 48 if SMOKE else (800 if QUICK else None)
TEST_LIMIT = 48 if SMOKE else (1000 if QUICK else None)
EPOCHS = 1 if SMOKE else (3 if QUICK else 12)
train_ds, val_ds, test_ds, class_names = cu.fashion_data(
    train_limit=TRAIN_LIMIT, val_limit=VAL_LIMIT, test_limit=TEST_LIMIT)
train_loader = cu.loader(train_ds, batch_size=64, shuffle=True)
val_loader = cu.loader(val_ds, batch_size=64)
test_loader = cu.loader(test_ds, batch_size=64)
print("Class order:", class_names)


## Step 2 — Train and freeze the final model choice

The helper restores the epoch with lowest validation cross-entropy. After choosing the model,
evaluate on test once. Do not change hyperparameters in response to the test score.
The quick subset and short run are a teaching experiment, not a benchmark submission.


In [ ]:
cu.seed_everything(42)
model = cu.SmallCNN(num_classes=len(class_names)).to(DEVICE)
history = cu.fit(model, train_loader, val_loader, epochs=EPOCHS, device=DEVICE)
cu.plot_history(history)
test_metrics = cu.evaluate(model, test_loader, DEVICE)
print("Final test metrics:", test_metrics)


## Step 3 — Create a versioned model bundle

Save a `state_dict` of tensors rather than a pickled model object. Record architecture and preprocessing
in JSON. We load with `weights_only=True` later. Use only model artifacts from sources you trust.
The bundle name distinguishes smoke checks from demonstration models so a tiny validation run
does not replace the model you trained for class.


In [ ]:
bundle_dir = cu.ARTIFACTS / ("12_bundle_smoke" if SMOKE else "12_bundle_demo")
bundle_dir.mkdir(parents=True, exist_ok=True)
cpu_model = copy.deepcopy(model).cpu().eval()
torch.save(cpu_model.state_dict(), bundle_dir / "weights.pt")
metadata = {
    "schema_version": 1,
    "architecture": "SmallCNN",
    "in_channels": 1,
    "class_names": class_names,
    "preprocessing": {
        "color_mode": "L", "size_hw": [28, 28], "interpolation": "bilinear",
        "scale": "uint8_to_0_1", "mean": [0.0], "std": [1.0]
    },
    "dataset": "FashionMNIST",
    "training_examples": len(train_ds), "epochs_run": EPOCHS,
    "seed": 42, "smoke_only": SMOKE,
    "torch_version": str(torch.__version__),
    "test_metrics": test_metrics,
    "intended_use": "Educational classification of Fashion-MNIST-like clothing images",
    "limitations": ["Short demo training", "Not validated on ordinary clothing photographs",
                   "Softmax confidence is uncalibrated", "No unknown-object class"]
}
(bundle_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved bundle:", bundle_dir)


## Step 4 — Reload from disk and verify numerical parity

Construct a new model using only the documented architecture and metadata. This catches a
wrong class count, missing state, or different architecture before deployment.
Use a tolerance for floating-point comparisons instead of assuming bit-for-bit equality across devices.


In [ ]:
from predict_image import load_bundle, predict_image

loaded_model, loaded_metadata, preprocess = load_bundle(bundle_dir, device="cpu")
sample_images, sample_labels = next(iter(test_loader))
with torch.inference_mode():
    before = cpu_model(sample_images)
    after = loaded_model(sample_images)
torch.testing.assert_close(before, after)
print("Reloaded model produces matching logits.")


## Step 5 — Predict one actual image file

Save a test image as PNG, then pass its file path through the same public prediction function the
command-line script uses. The example already matches the training distribution; supplying an
unrelated photograph is a distribution-shift experiment, not a supported use case.


In [ ]:
example_image, example_label = test_ds[0]
example_path = bundle_dir / "example.png"
transforms.ToPILImage()(example_image).save(example_path)
result = predict_image(bundle_dir, example_path, device="cpu", top_k=3)
print(json.dumps(result, indent=2))
plt.imshow(Image.open(example_path), cmap="gray")
plt.title(f"True: {class_names[example_label]} | Predicted: {result['predictions'][0]['class_name']}")
plt.axis("off")
plt.show()


## Step 6 — Verify the file preprocessing contract

The saved PNG should reproduce the original tensor within quantization tolerance. This round-trip
test checks the complete input path, including conversion, resizing, and normalization.


In [ ]:
with Image.open(example_path) as pil_image:
    recovered_tensor = preprocess(pil_image)
torch.testing.assert_close(recovered_tensor, example_image, atol=1/255 + 1e-7, rtol=0)
assert result["predictions"][0]["class_index"] == int(after[0].argmax())
assert 0 <= result["predictions"][0]["probability"] <= 1
print("Image preprocessing and top prediction agree.")


## Step 7 — Run the standalone command-line predictor

This starts another Python process, so it cannot accidentally use a model left in the notebook's
memory. The script depends on this course's `course_utils.py`, PyTorch, torchvision, NumPy, and Pillow.
It accepts CPU or CUDA, validates the supported metadata schema, and returns JSON.


In [ ]:
import subprocess
command = [sys.executable, str(ROOT / "predict_image.py"),
           "--bundle", str(bundle_dir), "--image", str(example_path), "--device", "cpu"]
completed = subprocess.run(command, check=True, capture_output=True, text=True)
cli_result = json.loads(completed.stdout)
assert cli_result["predictions"][0]["class_index"] == result["predictions"][0]["class_index"]
print(completed.stdout)


## Step 8 — Export a fixed-shape inference program

`torch.export` captures a model program with input constraints. This teaching artifact accepts
**one float32 grayscale 28×28 image**, shape `[1, 1, 28, 28]`; larger batches are outside this contract.
It still needs a compatible PyTorch runtime. Export is not automatically a speed optimization or a
standalone executable. Other deployment targets may require additional conversion and testing.


In [ ]:
example_input = sample_images[:1].cpu().contiguous()
exported = torch.export.export(cpu_model, (example_input,))
export_path = bundle_dir / "classifier.pt2"
torch.export.save(exported, export_path)
restored_program = torch.export.load(export_path)
with torch.inference_mode():
    eager_output = cpu_model(example_input)
    exported_output = restored_program.module()(example_input)
torch.testing.assert_close(eager_output, exported_output, rtol=1e-4, atol=1e-5)
print("Export saved and parity checked:", export_path)


## Step 9 — Measure model latency honestly

Warm up first, then synchronize CUDA around timing because GPU operations are asynchronous.
Report batch size and whether loading/preprocessing/device transfer are included. This experiment
measures only a model forward pass with the input already on its target device. A small model at
batch size one can be faster on CPU; measure instead of assuming a GPU always wins.


In [ ]:
def latency_ms(candidate, x, repeats=20):
    candidate.eval()
    timings = []
    with torch.inference_mode():
        for _ in range(5):
            candidate(x)
        if x.is_cuda:
            torch.cuda.synchronize()
        for _ in range(repeats):
            start = time.perf_counter()
            candidate(x)
            if x.is_cuda:
                torch.cuda.synchronize()
            timings.append((time.perf_counter() - start) * 1000)
    return {"p50_ms": float(np.percentile(timings, 50)),
            "p95_ms": float(np.percentile(timings, 95)), "batch_size": len(x)}

latencies = {"cpu": latency_ms(cpu_model, example_input)}
if DEVICE.type == "cuda":
    latencies["cuda"] = latency_ms(model, example_input.to(DEVICE))
print(json.dumps(latencies, indent=2))
(bundle_dir / "latency.json").write_text(json.dumps(latencies, indent=2), encoding="utf-8")


## Step 10 — Review the model card before reusing the artifact

The metadata contains intended use, class order, training settings, test metrics, and limitations.
For a real project also record dataset provenance and permission, group/time splitting decisions,
per-class results, calibration, failure cases, the input contract, and retraining triggers.
Monitor input quality and class-specific performance after deployment; a high original test score
does not establish performance under future distribution shift.


In [ ]:
print((bundle_dir / "metadata.json").read_text(encoding="utf-8"))
print("Bundle files:", [p.name for p in sorted(bundle_dir.iterdir())])


## Capstone — Build and demonstrate a classifier for your own task

Choose a small task with clearly defined classes, such as three types of household objects.
Start from lesson 07's folder pipeline, then adapt lesson 06's pretrained ResNet18.

| Milestone | Evidence to show in your demonstration |
|---|---|
| Define the task | Class definitions, intended use, representative example images |
| Audit the data | Counts, duplicates, corrupt images, group-aware or time-aware split rationale |
| Establish a baseline | Majority-class score and a simple trained model |
| Improve the model | One controlled validation comparison at a time |
| Diagnose errors | Confusion matrix, minority-class recall, representative mistakes |
| Assess reliability | Calibration split, confidence diagram, plausible corruption tests |
| Package prediction | Weights, label order, preprocessing, standalone image prediction |
| Final report | Once-only held-out test results and concrete limitations |

**Suggested 10-minute demonstration:** explain the problem (1 min), inspect data (2 min), run a
short training segment (2 min), compare results and mistakes (2 min), predict a new image (2 min),
and discuss one limitation (1 min). Keep a previously trained model for presentation continuity.

**Success criteria:** another person can restart the kernel, run every cell, reproduce your pipeline,
and understand what your model can and cannot classify.


## Exercises

1. Deliberately change the class order in a copied metadata file. Why can numerical parity still pass
   while the named prediction becomes wrong?
2. Try a batch of two images with the exported program. Explain its shape constraint.
3. Add a rule that flags images of the wrong aspect ratio. Is this enough to detect unknown classes?
4. For a real application, measure end-to-end latency from image-file read to returned JSON.

<details><summary>Suggested answers</summary>

1. Tensor values stay identical, but their class-index interpretation changes. Validate label mapping as part of the bundle.
2. The export example fixes batch size to one. Dynamic shapes need an explicitly designed and tested export contract.
3. No. Input validation catches malformed inputs; unknown-class detection is a separate problem requiring its own evaluation.
4. Start the timer before image loading and stop after prediction formatting; include warm-up policy and hardware in the report.

</details>

**Primary references:** [saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html),
[torch.export](https://docs.pytorch.org/docs/stable/export.html),
[benchmarking recipe](https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html).
